# Bengali card OCR: code walkthrough

**Run it:** pick the **Bengali Card OCR (.venv)** kernel (or `.venv/bin/python`) in the top-right kernel picker, then **Run All**. Setup is in the README.

This notebook calls the **same pipeline as the app** (`app.py`). It is a guided tour of one card going through it, not a separate OCR implementation or a training notebook.

**Route (about 15 minutes):** input → extraction → alignment → field crops → recognition and validation → explicit review → JSON export.

- Outputs contain patient images and values. **Clear all outputs before committing.** Exports go to the gitignored `data/work/`.
- **Run All never accepts or corrects a value.** Only clicking a review button records an operator action.
- No reference answers are used for inference. Current accuracy numbers are in `docs/RESULTS.md`.


## 1. Locate the project and keep inference local

**Code:** `run.sh`, `pipeline/pipeline.py`. Nothing is installed or downloaded here. A Python socket guard allows local Jupyter traffic and blocks outbound connections from this kernel; restart the kernel to remove it.


In [ ]:
from pathlib import Path
import os, sys, io, json, time, base64, hashlib, html, inspect, socket
from datetime import datetime, timezone

ROOT = next((p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "app.py").is_file() and (p / "pipeline/pipeline.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository folder.")
# Check BEFORE optional imports or the socket guard: the editor may remember a
# previously selected kernel even when this notebook requests .venv.
EXPECTED_PREFIX = (ROOT / ".venv").resolve()
if Path(sys.prefix).resolve() != EXPECTED_PREFIX:
    raise RuntimeError(
        "Wrong notebook kernel. Current interpreter: " + sys.executable + "\n"
        "In the notebook's TOP-RIGHT kernel picker, choose Select Another Kernel > "
        "Jupyter Kernel > Bengali Card OCR (.venv), or Python Environments > "
        + str(ROOT / ".venv/bin/python") + "\n"
        "Then Restart Kernel and Run All."
    )
import importlib.util
missing = [name for name in ("ipywidgets", "IPython", "pandas", "PIL", "cv2", "onnxruntime")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "The project kernel is selected, but dependencies are missing: " + ", ".join(missing)
        + "\nPrepare the environment in a TERMINAL: " + str(ROOT / ".venv/bin/python")
        + " -m pip install -r " + str(ROOT / "requirements.txt")
        + " -r " + str(ROOT / "requirements-notebook.txt")
        + "\nThen Restart Kernel and Run All. Installation is separate from offline execution."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                  STREAMLIT_BROWSER_GATHER_USAGE_STATS="false")

BLOCKED_NETWORK_ATTEMPTS = []
_original_connect = getattr(socket.socket.connect, "_walkthrough_original", socket.socket.connect)
_original_connect_ex = getattr(socket.socket.connect_ex, "_walkthrough_original", socket.socket.connect_ex)
def _local_address(address):
    return not isinstance(address, tuple) or address[0] in ("127.0.0.1", "::1", "localhost")
def _local_connect(sock, address):
    if not _local_address(address):
        BLOCKED_NETWORK_ATTEMPTS.append(str(address))
        raise OSError("Walkthrough kernel blocks outbound networking; local assets are required.")
    return _original_connect(sock, address)
def _local_connect_ex(sock, address):
    if not _local_address(address):
        BLOCKED_NETWORK_ATTEMPTS.append(str(address))
        raise OSError("Walkthrough kernel blocks outbound networking; local assets are required.")
    return _original_connect_ex(sock, address)
_local_connect._walkthrough_original = _original_connect
_local_connect_ex._walkthrough_original = _original_connect_ex
socket.socket.connect = _local_connect
socket.socket.connect_ex = _local_connect_ex

from PIL import Image, ImageDraw
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, FileLink
from pipeline.io_utils import load_upload
from pipeline.pipeline import extract_card
from pipeline.layout import iter_cells
from pipeline.recognize import normalize_value, to_western_digits
from pipeline.structured import syntax_ok
from pipeline.validate import validate_field
from pipeline.review import review_field, accept_routine
from pipeline.export import build_export_payload, save_export
from pipeline.data_paths import PAGES, WORK

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = WORK / "notebook" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
for kind in ("EPI", "ANC", "NCD"):
    assert (ROOT / "assets/registration" / f"{kind}.png").is_file(), f"Missing {kind} alignment template (see README)"
assert (ROOT / "models/ppocr-cards/ppocrv6_cards.onnx").is_file(), "Missing the recognizer model"
print("Project:", ROOT)
print("Interpreter:", sys.executable)
print("Local export folder:", RUN_DIR)
print("Project kernel and notebook dependencies: ready")

## 2. Choose a card image or a PDF page

Default: **ANC page 058** from `data/pages/`. For EPI try `page-003.jpg`, for NCD `page-095.jpg`. For your own image or PDF, set `INPUT_PATH` to its path. `PDF_PAGE` is one-based. PDFs are rendered by the same `load_upload()` as the app; only the selected page goes to OCR.

To use another input, change this cell and **Restart Kernel → Run All**. That clears in-memory reviews, so export real reviews first.

**Code:** `pipeline/io_utils.py::load_upload`, `app.py::main`.


In [ ]:
INPUT_PATH = PAGES / "page-058.jpg"
PDF_PAGE = 1
CARD_TYPE_OVERRIDE = None       # None = automatic; otherwise "EPI", "ANC", or "NCD"
MANUAL_CLOCKWISE_ROTATION = 0   # 0, 90, 180, 270
RUN_ALL_CARD_TYPES = True
RUN_WARM_TIMING = False         # optional repeated calls (warm timings only)

INPUT_PATH = Path(INPUT_PATH).expanduser().resolve()
input_tick = time.perf_counter()
pages, page_labels = load_upload(INPUT_PATH.read_bytes(), INPUT_PATH.name)
if not 1 <= PDF_PAGE <= len(pages):
    raise ValueError(f"PDF_PAGE must be between 1 and {len(pages)}")
source_image = pages[PDF_PAGE - 1]
source_label = page_labels[PDF_PAGE - 1]
input_decode_seconds = time.perf_counter() - input_tick
preview = source_image.copy()
preview.thumbnail((1100, 750))
display(Markdown(f"**Original input:** `{source_label}` · page {PDF_PAGE}/{len(pages)} · {source_image.size}"))
display(preview)
print(f"Input read/decode: {input_decode_seconds:.4f}s")

## 3. Run the actual application pipeline

This is the **same `extract_card()` call used by `app.py`**. It loads/rotates pixels, registers a layout, crops fields, runs local OCR, validates candidates and returns provenance/timings. No predictions are populated from filenames or saved labels.

This first-call wall time is for this kernel with input already decoded above, not a full cold app or browser measurement.

**Code:** `pipeline/pipeline.py::extract_card`.

In [ ]:
inference_tick = time.perf_counter()
result = extract_card(source_image, card_type_override=CARD_TYPE_OVERRIDE,
                      clockwise_degrees=MANUAL_CLOCKWISE_ROTATION)
result["source"] = source_label
notebook_inference_seconds = time.perf_counter() - inference_tick
aligned_image = result["oriented_image"]
field_map = {field["field_id"]: field for field in result["fields"]}
assert len(field_map) == len(result["fields"]), "Duplicate schema field IDs"
assert all(field["needs_review"] for field in result["fields"])
print(f"Detected {result['card_type']} · {len(field_map)} configured fields · {notebook_inference_seconds:.3f}s")
print("Every field is pending review. Field count is not recognition accuracy.")

## 4. Classification, orientation and alignment

SIFT finds visual landmarks; RANSAC rejects inconsistent matches; the homography warps the card to canonical coordinates. Classification and automatic orientation share this registration step. This is not a general AI box detector or a title-reading classifier.

The override narrows template choices; it does not bypass quality gates. The homography maps the input **after manual rotation** into aligned-page pixels. A poor/unknown layout raises an error instead of populating arbitrary fields.

**Code:** `pipeline/alignment.py::_features`, `_template`, `register_page`.

In [ ]:
registration = result["preprocess"]["alignment"]
display(pd.DataFrame([{key: attempt.get(key) for key in
                      ("card_type", "accepted", "matches", "inliers", "inlier_ratio", "reason")}
                     for attempt in registration["attempts"]]))
print("Canonical size:", aligned_image.size)
print("Clockwise rotation metadata:", result["preprocess"]["clockwise_degrees"])
print("Homography input frame:", registration["homography_input_frame"])
display(pd.DataFrame(registration["homography"], columns=["H0", "H1", "H2"]))
preview = aligned_image.copy()
preview.thumbnail((1100, 750))
display(Markdown("**Aligned canonical image** — field coordinates refer to this frame."))
display(preview)

## 5. Field and visit mapping

Known-layout geometry comes from `config/layout_templates.json`; OCR supplies candidate values. ANC has **six maternal visit columns and three delivery columns**. NCD visits are rows. EPI uses vaccine rows and dose positions.

The current `geometry="inside"` uses schema boxes plus native per-column polygons for ANC height. The original review crop stays unmodified; only recognition input is masked/prepared. Generic border snapping was measured and rejected as the default because it made ANC agreement worse.

**Code:** `pipeline/layout.py::iter_cells`; `pipeline/grid.py::merged_anc_height_quad`, `masked_polygon_crop`.

In [ ]:
# Inspection only: regenerate descriptors using the same layout function/mode as extract_card.
# These do not replace the predictions in result.
cells = iter_cells(aligned_image, result["card_type"], geometry="inside")
cell_map = {cell["field_id"]: cell for cell in cells}
assert set(cell_map) == set(field_map)
fields_table = pd.DataFrame([{key: field.get(key) for key in
                             ("field_id", "printed_label_en", "section", "row_index", "column_index",
                              "column_group", "visit_index", "delivery_column_index", "value_type",
                              "value_bn_raw", "status", "needs_review")}
                            for field in result["fields"]])
display(fields_table.groupby(["section", "column_group"], dropna=False).size().rename("configured slots").to_frame())
display(fields_table.head(15))
preferred_key = {"ANC": "glucose", "NCD": "weight", "EPI": "bcg"}[result["card_type"]]
focus = next((field for field in result["fields"] if field["field_key"] == preferred_key), result["fields"][0])
FOCUS_FIELD_ID = focus["field_id"]
print("Starting review field:", FOCUS_FIELD_ID)

## 6. Inspect the pixels the recognizer receives

The original source crop, line-cleaned crop and tight recognition crop are different artifacts. Long printed rules and template-relative ink evidence help reduce interference, but can remove faint strokes or strokes touching a border. A detected blank is also unverified.

No patient numbers are repaired from neighboring fields or clinical context. Border-overlapping ANC height remains especially uncertain.

**Code:** `pipeline/cellprep.py::prepare_cell`; `pipeline/recognize.py::BlankAwareEngine.recognize_cell`.

In [ ]:
focused_cell = cell_map[FOCUS_FIELD_ID]
for title, key in (("Original aligned source crop", "crop"),
                   ("Line-cleaned / polygon-masked recognition region", "clean_crop"),
                   ("Actual tight input passed to the cell recognizer", "tight_crop")):
    display(Markdown(f"**{title}**"))
    image = focused_cell[key].copy()
    image.thumbnail((1100, 300))
    display(image)
print("Field bounding box:", focused_cell["bbox_xyxy"])
print("Actual review-crop bounds:", focused_cell["crop_bbox_xyxy"])
print("Image-only ink evidence:", focused_cell["ink_evidence"])
print("Geometry:", focused_cell["crop_geometry"])

## 7. Raw observations, normalization and validation

`value_bn_raw` means **decoded model observation**, not a guarantee of faithful handwriting. Keep it separate from normalized convenience values. The synthetic BP example below demonstrates representation only; it is not a transcription of this patient.

“Readable” means syntax is plausible. Confidence is uncalibrated. Neither status nor score approves a field. Dates/BP/marks have syntax checks. For two-line EPI dates whose character-by-character read is not a real date (month 00, day 35, a dropped dot), the recognizer keeps the most likely valid date under the model's own probabilities. Those reads are always flagged `check`, and the warning quotes the raw read.

**Code:** `normalize_value`, `to_western_digits`, `pipeline/structured.py::syntax_ok`, `pipeline/validate.py::validate_field`, `pipeline/date_grammar.py::read_two_line_date`, `pipeline/card_logic.py::review_priority`.

In [ ]:
field = field_map[FOCUS_FIELD_ID]
display(pd.DataFrame([{key: field.get(key) for key in
                      ("field_id", "value_bn_raw", "normalized", "normalized_western", "status",
                       "confidence", "confidence_calibrated", "script_fidelity", "review_priority", "needs_review")}]))
display(pd.DataFrame(field["candidates"]))
print("Warnings:", *field["warnings"], sep="\n- ")
print("Syntax check:", syntax_ok(field.get("value_bn_raw"), field["value_type"]))
print("Validation:", validate_field(field.get("value_bn_raw"), field["value_type"], field["status"]))
illustrative_bp = "১২০/৮০"  # synthetic demonstration only
print("Synthetic example:", illustrative_bp, "→", to_western_digits(illustrative_bp))

## 8. Explicit field review — select, inspect, Accept or Correct

**Run All does not click these buttons.** Inspect the source before using them. `Accept OCR value` keeps the model observation; `Correct value` saves exactly what you type, including an intentional empty string. Pending unknowns remain `null` in the final export. Do not enter a convenient answer to improve the demonstration.

**Accept all routine fields** is the same one-click action as the app. It accepts only unreviewed fields marked `routine` (blanks, marks, or valid values above the score cutoff chosen on validation pages), and each is logged as `explicit_operator_bulk_accept_routine`. Check fields are marked ⚠ and listed first.

Changing fields retains confirmed entries in the `reviews` dictionary. Rerunning this cell clears those in-memory reviews. Export them first if needed. Simulated notebook tests are not independent human gold labels.

**Code:** the same `pipeline/review.py::review_field`, `accept_routine` and `pipeline/export.py::build_export_payload` used by the app. If widgets are unavailable in your editor, select the project kernel and enable its Jupyter widget support; do not bypass review with automatic approvals.

In [ ]:
reviews = {}
# Check fields first, as in the app. Labels stay fixed after review so the picker keeps its choice.
field_picker = widgets.Dropdown(
    options=[(("⚠ " if f.get("review_priority") != "routine" else "")
              + f"{f['field_id']} · {f.get('printed_label_en') or f['field_key']}", f["field_id"])
             for f in sorted(result["fields"], key=lambda f: f.get("review_priority") == "routine")],
    value=FOCUS_FIELD_ID,
    description="Field:", layout=widgets.Layout(width="95%"))
reviewed_text = widgets.Textarea(description="Reviewed:", layout=widgets.Layout(width="95%", height="75px"))
accept_button = widgets.Button(description="Accept OCR value", button_style="success")
correct_button = widgets.Button(description="Correct value", button_style="warning")
reset_button = widgets.Button(description="Reset review")
bulk_button = widgets.Button(description="Accept all routine fields", layout=widgets.Layout(width="220px"))
review_output = widgets.Output()
review_summary = widgets.HTML()

def show_review_field(change=None):
    selected = field_map[field_picker.value]
    rev = reviews.get(selected["field_id"], {})
    reviewed_text.value = rev.get("edited_value", selected.get("value_bn_raw") or "")
    accept_button.disabled = selected.get("value_bn_raw") is None
    payload_now = build_export_payload(result, reviews)
    check_left = sum(f.get("review_priority") != "routine" and f["field_id"] not in reviews for f in result["fields"])
    routine_left = sum(f.get("review_priority") == "routine" and f["field_id"] not in reviews
                       and f.get("value_bn_raw") is not None for f in result["fields"])
    review_summary.value = (f"<b>Reviewed:</b> {payload_now['review_summary']['reviewed']} / {len(field_map)} · "
                            f"<b>needs a look (check):</b> {check_left} · <b>routine not yet accepted:</b> {routine_left}. "
                            "Typing alone approves nothing. Export after review using the next cell's button.")
    review_output.clear_output(wait=True)
    with review_output:
        display(Markdown(f"**{selected['field_id']}** · {selected.get('column_group')} · "
                         f"row {selected['row_index']} / column {selected['column_index']}"))
        highlighted = aligned_image.copy()
        pen = ImageDraw.Draw(highlighted)
        if selected.get("field_polygon_xy"):
            pen.polygon([tuple(p) for p in selected["field_polygon_xy"]], outline="red", width=4)
        else:
            pen.rectangle(selected["bbox_xyxy"], outline="red", width=4)
        highlighted.thumbnail((1000, 650))
        display(highlighted)
        display(cell_map[selected["field_id"]]["crop"])
        display(pd.DataFrame([{"OCR observation": selected.get("value_bn_raw"),
                               "model status": selected["status"],
                               "priority": selected.get("review_priority", "check"),
                               "review status": rev.get("review_status", "pending"),
                               "saved reviewed value": rev.get("edited_value")}]))
        print("Warnings:", *selected["warnings"], sep="\n- ")

def accept_selected(_):
    selected = field_map[field_picker.value]
    reviews[selected["field_id"]] = review_field(selected, reviewed_text.value, "accept", result["source_sha256"])
    show_review_field()

def correct_selected(_):
    selected = field_map[field_picker.value]
    reviews[selected["field_id"]] = review_field(selected, reviewed_text.value, "correct", result["source_sha256"])
    show_review_field()

def accept_all_routine(_):
    accept_routine(result["fields"], reviews, result["source_sha256"])
    show_review_field()

def reset_selected(_):
    reviews.pop(field_picker.value, None)
    show_review_field()

field_picker.observe(show_review_field, names="value")
accept_button.on_click(accept_selected)
correct_button.on_click(correct_selected)
reset_button.on_click(reset_selected)
bulk_button.on_click(accept_all_routine)
display(widgets.VBox([field_picker, reviewed_text,
                      widgets.HBox([accept_button, correct_button, reset_button, bulk_button]), review_summary, review_output]))
show_review_field()

## 9. Export model predictions separately from operator decisions

The first export below deliberately contains pending final values. After explicit review, click **Export current review JSON** to write a new, uniquely named local file. Existing exports are never overwritten.

Each payload includes mapping, source hash, transform/model metadata, raw observation, normalized representations and human review action/time. An accepted blank is `""`; a pending final value is `null`. This is operator provenance, not authenticated clinical authority. Notebook export is separate from the Streamlit download button, but uses the **same payload builder and serializer**.

**Code:** `pipeline/export.py::build_export_payload`, `save_export`.

In [ ]:
export_output = widgets.Output()
export_button = widgets.Button(description="Export current review JSON", layout=widgets.Layout(width="240px"))
exported_paths = []

def export_current_reviews(_=None):
    payload = build_export_payload(result, reviews)
    # Operator review must never mutate the original model observations.
    for original, exported in zip(result["fields"], payload["fields"]):
        assert original.get("value_bn_raw") == exported["model_prediction"]["value_bn_raw"]
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    path = save_export(payload, RUN_DIR / f"{result['card_type']}_{stamp}.json")
    exported_paths.append(path)
    export_output.clear_output(wait=True)
    with export_output:
        print("Local JSON:", path)
        print("Review summary:", payload["review_summary"])
        display(FileLink(os.path.relpath(path, Path.cwd())))
        selected_export = next(f for f in payload["fields"] if f["field_id"] == field_picker.value)
        print(json.dumps(selected_export, ensure_ascii=False, indent=2))
    return path

export_button.on_click(export_current_reviews)
display(export_button, export_output)
initial_payload = build_export_payload(result, reviews)
if not reviews:
    assert all(f["needs_review"] and f["final_value"] is None for f in initial_payload["fields"])
initial_export = export_current_reviews()

## 10. One card of each type: EPI, ANC and NCD

The same pipeline on one sample of each card type. Readings are not promised to be correct. Two-line EPI dates are the weakest area, and reads changed by the date rule are listed below; each is flagged check. Page 003 was used for tuning, so it is not an unseen example. Set `RUN_ALL_CARD_TYPES=False` to skip.


In [ ]:
all_card_results = {}
if RUN_ALL_CARD_TYPES:
    for expected_kind, number in (("EPI", 3), ("ANC", 58), ("NCD", 95)):
        sample_path = PAGES / f"page-{number:03}.jpg"
        sample_result = extract_card(sample_path)
        assert sample_result["card_type"] == expected_kind
        assert all(f["needs_review"] for f in sample_result["fields"])
        all_card_results[expected_kind] = sample_result
    display(pd.DataFrame([{"page": Path(r["source"]).name, "type": kind,
                           "configured fields": len(r["fields"]), "pipeline seconds": r["timing_sec"],
                           "requires review": r["stats"]["n_needs_review"],
                           "check": sum(f.get("review_priority") != "routine" for f in r["fields"]),
                           "routine": sum(f.get("review_priority") == "routine" for f in r["fields"]),
                           "date-rule reads": sum("ocr_source=date_grammar" in f["warnings"] for f in r["fields"])}
                          for kind, r in all_card_results.items()]))
    date_rule_rows = [{"page": Path(r["source"]).name, "field_id": f["field_id"],
                       "value_bn_raw": f["value_bn_raw"], "priority": f.get("review_priority"),
                       "warning": next(w for w in f["warnings"] if w.startswith("Most likely valid date"))}
                      for r in all_card_results.values() for f in r["fields"]
                      if "ocr_source=date_grammar" in f["warnings"]]
    display(Markdown("**Two-line dates replaced by the most likely valid date (all flagged check):**"))
    display(pd.DataFrame(date_rule_rows) if date_rule_rows else Markdown("_None on these pages._"))
else:
    print("Three-card demonstration skipped by configuration.")

## 11. Timing

Stages below come from the run in step 3. Source decoding happened in step 2, and Jupyter rendering is not the app's rendering. Optional repeated timing is warm-only, because earlier cells already loaded the model.

For accuracy, see `docs/RESULTS.md`. Human-label scoring is `evaluation/verified_accuracy.py`.


In [ ]:
stage_table = pd.DataFrame(list(result["stage_timing_sec"].items()), columns=["stage", "seconds"])
display(stage_table)
print(f"Separate input read/decode: {input_decode_seconds:.4f}s; notebook inference wall: {notebook_inference_seconds:.4f}s")
if RUN_WARM_TIMING:
    warm_rows = []
    for repetition in range(3):
        start = time.perf_counter()
        extract_card(source_image, card_type_override=CARD_TYPE_OVERRIDE,
                     clockwise_degrees=MANUAL_CLOCKWISE_ROTATION)
        warm_rows.append({"warm repetition": repetition + 1, "wall seconds": time.perf_counter() - start})
    display(pd.DataFrame(warm_rows))

current_payload = build_export_payload(result, reviews)
assert all(original["value_bn_raw"] == exported["model_prediction"]["value_bn_raw"]
           for original, exported in zip(result["fields"], current_payload["fields"]))
WALKTHROUGH_RUN_REPORT = {
    "source": source_label, "card_type": result["card_type"], "fields": len(field_map),
    "three_card_types_run": list(all_card_results),
    "reviewed_fields": current_payload["review_summary"]["reviewed"],
    "pending_fields": current_payload["review_summary"]["pending"],
    "exports_written": len(exported_paths), "blocked_python_socket_attempts": len(BLOCKED_NETWORK_ATTEMPTS),
    "source_code_reused": "pipeline.pipeline.extract_card",
    "verified_accuracy": None,
}
print("WALKTHROUGH_RUN_REPORT=" + json.dumps(WALKTHROUGH_RUN_REPORT))


## 12. Where the code is

| File | Function/class | What it does |
|---|---|---|
| `app.py` | `main`, `clear_review` | Input, state isolation, review and download |
| `pipeline/pipeline.py` | `extract_card` | One orchestration path shared by app and notebook |
| `pipeline/alignment.py` | `register_page` | SIFT/RANSAC alignment; refuses a card it cannot match |
| `pipeline/layout.py` | `iter_cells` | Field identity, row/column groups and crops |
| `pipeline/grid.py` | `merged_anc_height_quad`, `masked_polygon_crop` | Sloping merged ANC height field |
| `pipeline/cellprep.py` | `prepare_cell` | Original versus prepared input, and blank evidence |
| `pipeline/recognize.py` | `PPCardsEngine`, `BlankAwareEngine` | Model reads, candidate routing and uncertainty; digits-only decoding |
| `pipeline/marks.py` | `classify_mark` | Tick/cross/(±) reader; a possible "+" always goes to a human |
| `pipeline/date_grammar.py` | `read_two_line_date` | Most likely valid EPI date when the raw read is impossible; always flagged check |
| `pipeline/structured.py` | `format_structured` | Date/BP separators only when exactly one valid value fits |
| `pipeline/card_logic.py` | `epi_same_day_columns`, `review_priority` | Same-day EPI suggestion (flagged) and routine/check priority |
| `pipeline/review.py` | `review_field`, `accept_routine` | Explicit action, timestamp and source provenance |
| `pipeline/export.py` | `build_export_payload` | Pending final values versus reviewed records |
| `training/finetune_ppocrv6.py` | `main` | How the model was trained: synthetic digits first, then card crops |
| `evaluation/verified_accuracy.py` | `evaluate` | Scores human labels against the model |

The source viewer below reads the live imported function, not a pasted copy.


In [ ]:
source_text = inspect.getsource(extract_card)
source_path = Path(inspect.getsourcefile(extract_card)).resolve()
source_line = inspect.getsourcelines(extract_card)[1]
display(HTML("<details><summary>Actual extract_card source: "
             + html.escape(str(source_path.relative_to(ROOT))) + f" (line {source_line})</summary><pre>"
             + html.escape(source_text) + "</pre></details>"))

## Summary

**Pipeline:** image/PDF → alignment and card-type detection → field crops → local OCR → validation and uncertainty flags → explicit operator review → JSON.

**Model:** PP-OCRv6_small_rec (Apache-2.0), pretrained on synthetic cells built from NumtaDB handwritten Bengali digits (CC BY-SA 4.0), then fine-tuned on card crops. CPU, ONNX, 12 MB.

**Limits:** two-line EPI dates are the weakest field type; phone photos are not measured; cards come from one clinic's writers; accuracy against human labels is below the 95% target (see `docs/RESULTS.md`).

**Run the app:** `./run.sh` → http://127.0.0.1:8501.

**Before committing this notebook:** clear all outputs. Restarting the kernel clears reviews and the socket guard.
